In [1]:
import pandas as pd
import numpy as np
import re

from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw/classification")

garmin_path = DATA_DIR / "Garmin Connect.en.csv"
huawei_path = DATA_DIR / "Huawei Health.en.csv"
samsung_path = DATA_DIR / "Samsung Health.en.csv"

In [5]:
garmin = pd.read_csv(garmin_path)
huawei = pd.read_csv(huawei_path)
samsung = pd.read_csv(samsung_path)

print("Garmin:", garmin.shape)
print("Huawei:", huawei.shape)
print("Samsung:", samsung.shape)

Garmin: (2000, 8)
Huawei: (2000, 8)
Samsung: (2000, 8)


In [4]:
garmin["app"] = "Garmin Connect"
huawei["app"] = "Huawei Health"
samsung["app"] = "Samsung Health"

In [6]:
df = pd.concat(
    [garmin, huawei, samsung],
    ignore_index=True
)

print("Combined dataset shape:", df.shape)

Combined dataset shape: (6000, 8)


In [7]:
df.columns.to_list()

['data',
 'score',
 'thumbsUpCount',
 'problem_report',
 'feature_request',
 'irrelevant',
 'ori_lang',
 'app']

In [8]:
df = df[
    [
        "data",
        "problem_report",
        "feature_request",
        "irrelevant",
        "app"
    ]
].copy()

In [9]:
df.head()

,data,problem_report,feature_request,irrelevant,app
0,Awesome! I thought I hated running but the gar...,0,0,1,Garmin Connect
1,On galaxy s4,0,0,1,Garmin Connect
2,Vivo Active HR active/intensity minutes have s...,1,0,0,Garmin Connect
3,After last update how i can see my scheduled t...,1,0,0,Garmin Connect
4,Good overall. Could do with a couple of extra ...,0,1,0,Garmin Connect


In [ ]:
# Missing values
print("Missing review text:", df["data"].isna().sum())
print("Empty review text:", (df["data"].fillna("").str.strip() == "").sum())

Missing review text: 0
Empty review text: 0


In [ ]:
# Checking Ambiguous labels
label_count = df[
    ["problem_report", "feature_request", "irrelevant"]
].sum(axis=1)

print("Single-label reviews:", (label_count == 1).sum())
print("Multi-label reviews:", (label_count > 1).sum())
print("Unlabeled reviews:", (label_count == 0).sum())

Single-label reviews: 5895
Multi-label reviews: 103
Unlabeled reviews: 2


In [12]:
df = df[label_count == 1].copy()

print("Dataset after label filtering:", df.shape)

Dataset after label filtering: (5895, 5)


In [32]:
# Creating target label
def create_label(row):
    if row["problem_report"] == 1:
        return "Bug Report"
    elif row["feature_request"] == 1:
        return "Feature Request"
    elif row["irrelevant"] == 1:
        return "General Feedback"
    
df["label"] = df.apply(create_label, axis=1)

In [33]:
df["label"].value_counts()

label
General Feedback    2986
Bug Report          1863
Feature Request     1046
Name: count, dtype: int64

In [34]:
df["data"] = df["data"].astype(str)

In [35]:
df["data"] = (
    df["data"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [36]:
# Removing Empty Reviews
df = df[df["data"].str.len() > 0].copy()

print("Final dataset size:", len(df))

Final dataset size: 5895


In [37]:
df[["data", "label", "app"]].head(10)

,data,label,app
0,Awesome! I thought I hated running but the gar...,General Feedback,Garmin Connect
1,On galaxy s4,General Feedback,Garmin Connect
2,Vivo Active HR active/intensity minutes have s...,Bug Report,Garmin Connect
3,After last update how i can see my scheduled t...,Bug Report,Garmin Connect
4,Good overall. Could do with a couple of extra ...,Feature Request,Garmin Connect
5,I don't know the point of this app if it canno...,Bug Report,Garmin Connect
6,Best set of tools for the money,General Feedback,Garmin Connect
7,This app and the watch help so much,General Feedback,Garmin Connect
8,love it xx,General Feedback,Garmin Connect
9,Great fitness companion and smart watch.,General Feedback,Garmin Connect


In [38]:
print(df["label"].value_counts())
print()
print(df["app"].value_counts())

label
General Feedback    2986
Bug Report          1863
Feature Request     1046
Name: count, dtype: int64

app
Samsung Health    1982
Garmin Connect    1971
Huawei Health     1942
Name: count, dtype: int64


In [39]:
PROCESSED_DIR = Path("../data/processed")
output_path = PROCESSED_DIR / "forcora_reviews.csv"

In [40]:
df[["data", "label", "app"]
].to_csv(output_path, index=False)

print(f"Saved processed dataset to: {output_path}")

Saved processed dataset to: ..\data\processed\forcora_reviews.csv
